In [1]:
%pip install -q nltk

Note: you may need to restart the kernel to use updated packages.


Converting the strings of text into a vocabulary index

In [235]:
import csv
import re
import nltk
from nltk.tokenize import RegexpTokenizer

nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('universal_tagset', quiet=True)

file_path = './propaganda_dataset_v2/propaganda_train.tsv'
silver_file_path = './propaganda_dataset_v2/propaganda_silver_augmented_master.tsv'

regex_tokenizer = RegexpTokenizer(r'\w+|[^\w\s]')
num_pattern = re.compile(r'^\d|\d$')

# 1. Define a normalization map for all sneaky quote variants
QUOTE_STANDARDIZATION_MAP = str.maketrans({
    '“': '"',  # Left curly double
    '”': '"',  # Right curly double
    '„': '"',  # Low double curly
    '‟': '"',  # High-reversed double
    '«': '"',  # Left guillemet
    '»': '"',  # Right guillemet
    '‘': "'",  # Left curly single
    '’': "'",  # Right curly single/apostrophe
    '‚': "'",  # Low single curly
    '‛': "'",  # High-reversed single
    '`': "'",  # Backtick
    '´': "'"   # Acute accent
})

# Gold
with open(file_path, mode='r', encoding='utf-8') as file:
    tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)

    vocab = {}
    row_count = 0

    for row in tsv_reader:
        if row['label'] != "not_propaganda":
            row_count += 1

            clean_text = re.sub(r'\s*<BOS>\s*|\s*<EOS>\s*', ' ', row['tagged_in_context']).strip()
            clean_text = clean_text.translate(QUOTE_STANDARDIZATION_MAP)
            clean_text = clean_text.lower()

            tokens = regex_tokenizer.tokenize(clean_text)
            token_pos_tags = nltk.pos_tag(tokens, tagset='universal')
            
            for word, pos in token_pos_tags:
                # print(pos)
                if pos == ".":
                    pos = "__PUNC__"

                if pos in ('NUM', "__PUNC__"):
                    word = pos

                if word not in vocab:
                    vocab[word] = {'pos': [pos], 'counter': 1}
                else:
                    vocab[word]['counter'] += 1
                    _ if pos in vocab[word]['pos'] else vocab[word]['pos'].append(pos)
        


    # Silver 
    with open(silver_file_path, mode='r', encoding='utf-8') as file:
        tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
        silver_row_count = 0

        for silver_row in tsv_reader:
            silver_row_count += 1

            silver_clean_text = re.sub(r'\s*<BOS>\s*|\s*<EOS>\s*', ' ', silver_row['silver_tagged_in_context']).strip()
            silver_clean_text = clean_text.lower()

            silver_tokens = regex_tokenizer.tokenize(silver_clean_text)
            silver_token_pos_tags = nltk.pos_tag(silver_tokens, tagset='universal')
            
            for silver_word, silver_pos in silver_token_pos_tags:
                if silver_word not in vocab:
                    pass
                else:
                    vocab[silver_word]['counter'] += 1
                    _ if pos in vocab[silver_word]['pos'] else vocab[silver_word]['pos'].append(silver_pos)

print("Num golden rows:", row_count)
print("Num silver rows:", silver_row_count)

print(vocab['administration'])
print(vocab['people'])

print('__PUNC__:', vocab['__PUNC__'])
print('NUM:     ', vocab['NUM'])

print('":       ', vocab['"'])
print("':       ", vocab["'"])

print("vocab BEFORE:", len(vocab))

rare_word_keys = [word for word, info in vocab.items() if info['counter'] == 1]
rare_words_dict = {word: vocab.pop(word) for word in rare_word_keys}
rare_words_dict

print("vocab AFTER:", len(vocab))
print("rare_words_dict:", len(rare_words_dict))

Num golden rows: 1291
Num silver rows: 1291
{'pos': ['NOUN'], 'counter': 32}
{'pos': ['NOUN'], 'counter': 126}
__PUNC__: {'pos': ['__PUNC__'], 'counter': 4881}
NUM:      {'pos': ['NUM'], 'counter': 463}
":        {'pos': ['ADP', 'NOUN', 'VERB', 'ADJ', 'X', 'ADV', 'CONJ', 'DET', 'PRON', 'PRT'], 'counter': 21788}
':        {'pos': ['PRT', 'VERB'], 'counter': 237}
vocab BEFORE: 6177
vocab AFTER: 3212
rare_words_dict: 2965


- These seems extreme but it makes sense: ZipF law
- Many of these are hapax legomena, since instances, and due to typos, variations or digital artifcats.
- Keeping them in may invoke "Curse of Dimensionality"
- Our training set is approx 2500 rows, this means for these hapax words, the index in the vector will be 0 for 2499 instances
- This extreme sparsity makes it incredibly hard for a simple linear model to learn meaningful weights without overfitting

In [217]:
unique_pos_tags = set([word['pos'][0] for word in rare_words_dict.values()])

unk_mapper = {}
indexer = 0

for _ in unique_pos_tags:
    unk_mapper[_] = indexer
    indexer += 1

for _ in rare_words_dict.keys():
    _pos = rare_words_dict[_]["pos"][0]
    _index = unk_mapper[_pos]

    rare_words_dict[_]["index"] = _index


vocab["'"]

{'pos': ['PRT'], 'counter': 41}

In [189]:
for i in vocab.items():
    print(i[0])

"
the
obama
administration
misled
american
people
and
congress
because
they
were
desperate
to
get
a
deal
with
iran
,
said
sen
.
hitler
annihilated
400
000
germans
who
handicapped
or
suffered
from
chronic
diseases
as
noted
above
at
this
point
literally
every
piece
of
so
-
called
evidence
put
by
authorities
then
mainstream
media
cannot
be
trusted
should
considered
until
proven
otherwise
his
account
was
suspended
for
violating
twitter
’
s
rules
relating
“
hateful
conduct
”
it
is
apparently
reference
what
islamic
texts
themselves
say
couple
events
past
week
itself
more
aggressive
military
actions
that
could
place
u
forces
in
harm
way
trump
has
expelled
russian
diplomats
sanctioned
than
[
]
but
authorized
commitment
syria
when
near
two
decades
afghanistan
have
failed
secure
nation
against
return
al
isis
?
outside
seekers
only
reason
why
there
general
interest
qur
an
among
non
muslims
seek
answer
question
whether
not
terrorism
we
must
ask
ourselves
hasn
t
happened
ve
caught
on
synodal
outrag

when processing tokens: 

if the token is in vocab then get the index and put a count +1 into the dense vector at position index


if not, is the token in rare_words: if so get the index and put a count +1 into the dense vecotr at position index. also map the word to its unk token if required for the method


if not in either, then pos_tag and check unk_mapper for index

In [176]:
vocab["."]

{'pos': ['.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',
  '.',